# Study 826 — Treasury Duration BAB — the teardown

The beta ladder, the Newey-West BAB *t*, the factor-regression alpha and residual beta, the 1,000-permutation placebo, the two-era robustness cut, the costed leveraged timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_etfs': 5, 'n_days': 3894, 'rows': 4147, 'fingerprint': 'e423191a2863', 'beta_shy': 0.123, 'beta_iei': 0.47, 'beta_ief': 0.911, 'beta_tlh': 1.402, 'beta_tlt': 2.094, 'bab_bps': 1.31, 't_nw': 2.5, 't_1s': 2.37, 'sharpe': 0.6, 'gross_sharpe': 0.6, 'lev_lo_bps': 2.0, 'lev_hi_bps': 0.69, 'welch_t': 1.18, 'beta_lo': 0.238, 'beta_hi': 1.864, 'gross_lev': 5.04, 'alpha_bps': 1.32, 't_alpha': 2.51, 'beta_resid': -0.007, 'placebo_obs': 1.31, 'placebo_mean': 2.573, 'placebo_sd': 0.728, 'placebo_p': 0.991, 'placebo_sigma': -1.74, 'placebo_draws': 1000, 'era_early_bps': 0.32, 'era_early_t': 0.49, 'era_early_n': 1760, 'era_late_bps': 2.13, 'era_late_t': 2.68, 'era_late_n': 2134, 'timer_1_gross': 1.31, 'timer_1_cost': 0.02, 'timer_1_borrow': 0.11, 'timer_1_net': 1.19, 'timer_1_t': 2.14, 'timer_1_sharpe': 0.55, 'timer_1_ann': 3.0, 'timer_5_gross': 1.31, 'timer_5_cost': 0.08, 'timer_5_borrow': 0.11, 'timer_5_net': 1.13, 'timer_5_t': 2.03, 'timer_5_sharpe': 0.52, 'timer_5_ann': 2.8, 'null_mean_t': 0.42, 'null_sd_t': 1.08, 'null_fire': 1, 'planted_t': 30.79, 'planted_welch': 15.1, 'planted_beta_resid': 0.006}

## The beta ladder — each ETF's trailing-252d beta to the duration factor

The equal-weight duration factor; betas rise monotonically with maturity.

In [2]:
print('  SHY %.3f  IEI %.3f  IEF %.3f  TLH %.3f  TLT %.3f'
      % (R['beta_shy'], R['beta_iei'], R['beta_ief'], R['beta_tlh'], R['beta_tlt']))

  SHY 0.123  IEI 0.470  IEF 0.911  TLH 1.402  TLT 2.094


## The headline — Frazzini-Pedersen BAB return

Long low-beta legs levered to unit beta, short high-beta legs, beta-neutral.

In [3]:
print(f"BAB          : {R['bab_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}  Sharpe = {R['sharpe']:.2f}")
print(f"levered legs : low-beta {R['lev_lo_bps']:+.2f} vs high-beta {R['lev_hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"factor reg   : alpha {R['alpha_bps']:+.2f} bps/day (NW t = {R['t_alpha']:+.2f}), "
      f"residual beta = {R['beta_resid']:+.3f} (beta-neutral)")
print(f"the cage     : beta_lo {R['beta_lo']:.3f} / beta_hi {R['beta_hi']:.3f} -> {R['gross_lev']:.2f}x gross leverage")

BAB          : +1.31 bps/day  NW(10) t = +2.50  one-sample t = +2.37  Sharpe = 0.60
levered legs : low-beta +2.00 vs high-beta +0.69 bps (Welch t = +1.18)
factor reg   : alpha +1.32 bps/day (NW t = +2.51), residual beta = -0.007 (beta-neutral)
the cage     : beta_lo 0.238 / beta_hi 1.864 -> 5.04x gross leverage


## Placebo — permute the returns into the SAME leverage cage (1,000 permutations)

Keep the beta-rank weights and the 1/β leverage; permute which ETF's return feeds each leg. A real beta signal ⇒ observed in the far *right* tail. Here it is the opposite — random assignment does *better*.

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f}  "
      f"({R['placebo_sigma']:+.2f}sigma vs placebo mean)")
print('=> the beta SIGNAL adds no value; the positive number is levered carry.')

observed +1.31 bps vs placebo mean +2.573 (sd 0.728) -> right-tail p = 0.99100  (-1.74sigma vs placebo mean)
=> the beta SIGNAL adds no value; the positive number is levered carry.


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1760): +0.32 bps  NW t = +0.49
2018-2026 (n=2134): +2.13 bps  NW t = +2.68


## The timer — can you get paid for it?

One-way cost × turnover of the *levered* weights; short leg pays 50 bps/yr borrow. (Financing the ~5x gross leverage at the short rate is **not** charged here — with `rf≈0` the levered carry is flattered; a realistic financing rate erodes the little that remains.)

In [6]:
for tag,g,c,b,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_borrow'],R['timer_1_net'],R['timer_1_t']),
                      ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_borrow'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5}: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f} + borrow {b:.2f}/day, t={t:+.2f})")

 1 bp: gross +1.31 -> net +1.19 bps/day (cost 0.02 + borrow 0.11/day, t=+2.14)
5 bps: gross +1.31 -> net +1.13 bps/day (cost 0.08 + borrow 0.11/day, t=+2.03)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted low-beta alpha with a beta-neutral book.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from duration_bab import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=826+s, n_days=1300))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0015, seed=826, n_days=1600))
print(f"planted (edge=0.0015): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}, residual beta = {planted['beta_resid']:+.3f}")

null (edge=0), 8 seeds: NW t mean +0.54 (sd 1.17), |t|>=2 in 0/8
planted (edge=0.0015): NW t = +30.79, Welch t = +15.10, residual beta = +0.006


## Verdict

- **Signal — None.** The Frazzini-Pedersen low-risk / BAB alpha does **not** replicate inside the Treasury curve. The book prints **+1.31 bps/day** (NW *t* = **+2.50**, right sign) — but the permutation placebo shows the beta sort earns *less* than a random assignment into the same 1/β cage (observed +1.31 vs placebo +2.57 bps, ~1.7σ into the left tail), and it is entirely a 2018–2026 phenomenon (*t* = +0.49 / +2.68). The 20-seed synthetic control recovers a *planted* alpha cleanly (*t* = +30.8, fires on 1/20 nulls, residual β ≈ 0), so the machinery is sound — the beta *signal* adds nothing; the positive number is mechanical levered carry.
- **Tradability — Mirage.** What remains is ~3%/yr net (net *t* ≈ +2.1 at 1–5 bps costs) but it rests on ~5x gross leverage financed at `rf≈0`, is beaten by random assignment, and lives in one era — a mirage, not a low-risk premium.